**Я использую для этого проекта FLORENCE-2**

Florence-2 умеет:
     ├──< CAPTION> - описание изображения                
     ├── < DETAILED_CAPTION> - детальное описание         │     ├── < MORE_DETAILED_CAPTION> - очень детальное       
     ├── < OD> - object detection (bounding boxes)        
     ├── < DENSE_REGION_CAPTION> - описание регионов      
     ├── < REGION_PROPOSAL> - предложение регионов        │
      │     └── < OCR> - распознавание текста на изображении                │


In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
local_dir = snapshot_download(repo_id="microsoft/Florence-2-base")


In [ ]:
import cv2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
pip install transformers==4.39.3 flash_attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.9 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.2
    Uninstalling transformers-4.56.2:
      Successfully uninstalled transformers-4.56.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.39.3 which is incompatible.


In [ ]:
import os
os.environ['HUGGINGFACE_HUB_CACHE'] = '/content/drive/MyDrive/hf_cache'

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/florence-2-base",
    trust_remote_code=True
).to(device)

processor = AutoProcessor.from_pretrained("microsoft/florence-2-base", trust_remote_code=True)

In [ ]:
from PIL import Image
import torch

image = Image.open("/content/image(16).png").convert("RGB")
inputs = processor(text="<DETAILED_CAPTION>", images=image, return_tensors="pt").to(device)

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

print(processor.batch_decode(generated_ids, skip_special_tokens=True)[0])

The image shows a screenshot of a simple button with a message that reads "When you click, a message should appear". The button is rectangular in shape and has a white background with a blue border. The text is written in a bold, black font and is centered on the button.


In [ ]:
cap = cv2.VideoCapture("/content/drive/MyDrive/Code-Gazer/bug_video.mp4")
frame_id = 0
captions = []
while True:
  ret,frame = cap.read()
  if not ret:
    break
  if frame_id % 30 == 0:
      image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
      inputs = processor(text="<CAPTION>", images=image,return_tensors="pt").to(device)
      with torch.no_grad():
        ids = model.generate(**inputs,max_new_tokens=64)
      caption = processor.batch_decode(ids,skip_special_tokens=True)[0]
      captions.append(f"{frame_id/30:.2f}s : {caption}")

  frame_id += 1

  cap.release()

print("\n".join(captions))


0.00s : A simple button on a dark blue background.


In [ ]:
cap = cv2.VideoCapture("/content/drive/MyDrive/Code-Gazer/bug_video.mp4")
frame_id = 0
captions = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % 30 == 0:  # берём 1 кадр в секунду
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        inputs = processor(text="<CAPTION>", images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=64)
        caption = processor.batch_decode(ids, skip_special_tokens=True)[0]
        captions.append(f"{frame_id/30:.2f}s : {caption}")

    frame_id += 1

cap.release()

print("\n".join(captions))


0.00s : A simple button on a dark blue background.
1.00s : A simple button on a dark blue background.
2.00s : A simple button on a dark blue background.
3.00s : A simple button on a dark blue background.
4.00s : A simple button on a dark blue background.
5.00s : A simple button on a dark blue background.
6.00s : A simple button on a dark blue background.
7.00s : A simple button on a dark blue background.
8.00s : A simple button on a dark blue background.
9.00s : A simple button on a dark blue background.
10.00s : A simple button on a dark blue background.


In [ ]:
import numpy as np


In [ ]:
cap = cv2.VideoCapture("/content/drive/MyDrive/Code-Gazer/bug_video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)

frame_id = 0
observations = []
prev_frame = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % 30 == 0:  # 1 кадр/сек
        timestamp = frame_id / fps
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # 1️⃣ DETAILED CAPTION (что видим)
        inputs = processor(
            text="<MORE_DETAILED_CAPTION>",  # ← БОЛЬШЕ ДЕТАЛЕЙ!
            images=image,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=128)  # ← БОЛЬШЕ ТОКЕНОВ

        caption = processor.batch_decode(ids, skip_special_tokens=True)[0]

        # 2️⃣ OBJECT DETECTION (где кнопка, где курсор)
        inputs_od = processor(
            text="<OD>",  # Object Detection
            images=image,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            ids_od = model.generate(**inputs_od, max_new_tokens=128)

        objects = processor.batch_decode(ids_od, skip_special_tokens=True)[0]

        # 3️⃣ OCR (что написано на кнопке)
        inputs_ocr = processor(
            text="<OCR>",
            images=image,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            ids_ocr = model.generate(**inputs_ocr, max_new_tokens=64)

        text_on_screen = processor.batch_decode(ids_ocr, skip_special_tokens=True)[0]

        # 4️⃣ FRAME DIFFERENCE (насколько изменился кадр)
        frame_diff = 0.0
        if prev_frame is not None:
            diff = cv2.absdiff(prev_frame, frame)
            frame_diff = np.mean(diff)

        prev_frame = frame.copy()

        # 5️⃣ СОБИРАЕМ OBSERVATION
        observation = {
            "timestamp": timestamp,
            "caption": caption,
            "objects": objects,
            "text": text_on_screen,
            "frame_change": frame_diff,
        }
        observations.append(observation)

        print(f"[{timestamp:.1f}s] Change: {frame_diff:.1f} | {caption}")

    frame_id += 1

cap.release()

# TEMPORAL REASONING (главная фишка!)
print("\n=== TEMPORAL ANALYSIS ===")

# Детектим момент клика (большое изменение кадра)
click_detected = False
click_timestamp = None

for i in range(1, len(observations)):
    curr = observations[i]
    prev = observations[i-1]

    # Если frame_change резко вырос = пользователь что-то сделал
    if curr["frame_change"] > 20:  # threshold (настрой)
        click_detected = True
        click_timestamp = curr["timestamp"]
        print(f"🖱️ USER ACTION at {click_timestamp:.1f}s (frame changed by {curr['frame_change']:.1f})")
        break

# Проверяем: изменилось ли ЧТО-НИБУДЬ после клика?
if click_detected:
    # Берём кадры до и после клика
    before_click = observations[i-1]
    after_click_frames = observations[i+1:i+4]  # следующие 3 секунды

    # Сравниваем descriptions
    captions_after = [obs["caption"] for obs in after_click_frames]

    print(f"\n📊 BEFORE click: {before_click['caption']}")
    print(f"📊 AFTER click:")
    for idx, cap in enumerate(captions_after):
        print(f"   +{idx+1}s: {cap}")

    # Проверка: изменились ли captions?
    if all(cap == before_click["caption"] for cap in captions_after):
        print("\n🚨 BUG DETECTED: UI did not change after user action!")
        print("   Expected: Some visual feedback (text, modal, navigation)")
        print("   Actual: Nothing happened (screen frozen)")
    else:
        print("\n✅ UI responded to user action")

[0.0s] Change: 0.0 | The image is a screenshot of a simple button on a dark blue background. The button is rectangular in shape and has a white text box in the center. The text box reads "Обзиная кноника / Simple button" in Russian. Below the text box, there is a message that reads "When you click, a message should appear". On the right side of the button, there are two small icons - one is a pink button and the other is a yellow button. The image appears to be a login screen for a website or application.
[1.0s] Change: 0.2 | The image is a screenshot of a simple button on a dark blue background. The button is rectangular in shape and has a white text box in the center. The text box reads "Обзиная кноника / Simple button" in Russian. Below the text box, there is a message that reads "When you click, a message should appear". On the right side of the button, there are two small icons - one is a pink button with the word "click" written on it and the other is a yellow button with a smile

In [ ]:
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')

In [ ]:
# на всякий случай если что то идет не так
for m in genai.list_models():
    print(m.name, "—", m.display_name)


models/embedding-gecko-001 — Embedding Gecko
models/gemini-2.5-pro-preview-03-25 — Gemini 2.5 Pro Preview 03-25
models/gemini-2.5-flash-preview-05-20 — Gemini 2.5 Flash Preview 05-20
models/gemini-2.5-flash — Gemini 2.5 Flash
models/gemini-2.5-flash-lite-preview-06-17 — Gemini 2.5 Flash-Lite Preview 06-17
models/gemini-2.5-pro-preview-05-06 — Gemini 2.5 Pro Preview 05-06
models/gemini-2.5-pro-preview-06-05 — Gemini 2.5 Pro Preview
models/gemini-2.5-pro — Gemini 2.5 Pro
models/gemini-2.0-flash-exp — Gemini 2.0 Flash Experimental
models/gemini-2.0-flash — Gemini 2.0 Flash
models/gemini-2.0-flash-001 — Gemini 2.0 Flash 001
models/gemini-2.0-flash-exp-image-generation — Gemini 2.0 Flash (Image Generation) Experimental
models/gemini-2.0-flash-lite-001 — Gemini 2.0 Flash-Lite 001
models/gemini-2.0-flash-lite — Gemini 2.0 Flash-Lite
models/gemini-2.0-flash-preview-image-generation — Gemini 2.0 Flash Preview Image Generation
models/gemini-2.0-flash-lite-preview-02-05 — Gemini 2.0 Flash-Lite Pr

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
import google.generativeai as genai
import base64
from io import BytesIO



# === SETUP GEMINI ===
genai.configure(api_key="")  # задайте свой api_key
gemini_model = genai.GenerativeModel('gemini-pro-latest')

# === HELPER: Image to PIL ===
def numpy_to_pil(frame):
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

# === STAGE 1: FLORENCE PERCEPTION ===
cap = cv2.VideoCapture("/content/drive/MyDrive/Code-Gazer/bug.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)

frame_id = 0
observations = []
prev_frame = None
key_frames = []  # Сохраним ключевые кадры для Gemini

print("🔍 STAGE 1: Florence-2 перцепция...\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % 30 == 0:  # 1 кадр/сек
        timestamp = frame_id / fps
        image = numpy_to_pil(frame)

        # Florence анализ
        inputs = processor(
            text="<MORE_DETAILED_CAPTION>",
            images=image,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=150)

        caption = processor.batch_decode(ids, skip_special_tokens=True)[0]

        # Frame difference
        frame_diff = 0.0
        if prev_frame is not None:
            diff = cv2.absdiff(prev_frame, frame)
            frame_diff = np.mean(diff)

        prev_frame = frame.copy()

        observation = {
            "timestamp": timestamp,
            "caption": caption,
            "frame_change": frame_diff,
            "frame": frame.copy()
        }
        observations.append(observation)

        # Сохраняем ключевые моменты
        if frame_diff > 0.05 or timestamp == 0:
            key_frames.append(observation)

        print(f"[{timestamp:.1f}s] Change: {frame_diff:.1f} | {caption[:100]}...")

    frame_id += 1

cap.release()

print(f"\n✅ Собрано {len(observations)} кадров, {len(key_frames)} ключевых\n")

# === STAGE 2: GEMINI TEMPORAL REASONING ===
print("Gemini анализ временной последовательности...")

# Подготовим данные для Gemini
timeline_text = ""
for obs in observations:
    timeline_text += f"[{obs['timestamp']:.1f}с] Изменение кадра: {obs['frame_change']:.2f}\n"
    timeline_text += f"Описание: {obs['caption']}\n\n"

# Промпт на русском
russian_prompt = f"""Ты — эксперт по тестированию UI. Проанализируй запись экрана с багом.

ВРЕМЕННАЯ ЛИНИЯ СОБЫТИЙ:
{timeline_text}

КОНТЕКСТ:
- Это видео демонстрирует баг в веб-приложении
- Пользователь взаимодействует с кнопкой "Click me"
- Ожидается: при клике должно появиться сообщение
- На видео показано, что происходит на самом деле

ТВОЯ ЗАДАЧА:
1. Определи момент времени, когда пользователь кликнул кнопку (подсказка: появляется курсор или изменяется frame_change)
2. Проанализируй, что произошло ПОСЛЕ клика в следующие 3-5 секунд
3. Сравни ожидаемое и фактическое поведение
4. Сделай вывод: есть ли баг?

ФОРМАТ ОТВЕТА (строго следуй структуре):

## 🔍 АНАЛИЗ

**Момент действия пользователя:** [время в секундах]
**Что сделал пользователь:** [описание действия]

**Ожидаемое поведение:**
[что должно было произойти]

**Фактическое поведение:**
[что произошло на самом деле]

## 🐛 ВЫВОД

**Баг обнаружен:** [ДА/НЕТ]
**Описание бага:** [краткое описание]
**Критичность:** [Высокая/Средняя/Низкая]

## 📝 РЕКОМЕНДАЦИИ

**Шаги для воспроизведения:**
1. [шаг 1]
2. [шаг 2]
...

**Возможная причина:**
[техническая гипотеза о причине бага]
"""

# Отправляем в Gemini
response = gemini_model.generate_content(russian_prompt)

print("=" * 60)
print(response.text)
print("=" * 60)

# STAGE 3
# визуальный анализ ключевых кадров Gemini

print("\n🎨 STAGE 3: Визуальный анализ ключевых кадров...\n")

# Берём кадр ДО клика и ПОСЛЕ клика
before_click = key_frames[0] if len(key_frames) > 0 else observations[0]
after_click = key_frames[-1] if len(key_frames) > 1 else observations[-1]

# Конвертируем в формат для Gemini
def frame_to_base64(frame_np):
    pil_img = numpy_to_pil(frame_np)
    buffered = BytesIO()
    pil_img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode()

visual_prompt = f"""Сравни два скриншота из видео:

КАДР 1 (до клика, {before_click['timestamp']:.1f}с):
Florence описание: {before_click['caption']}

КАДР 2 (после клика, {after_click['timestamp']:.1f}с):
Florence описание: {after_click['caption']}

Вопрос: Видишь ли ты ВИЗУАЛЬНЫЕ различия между кадрами?
Появилось ли какое-то новое сообщение, модальное окно, или изменился ли UI?

Ответь кратко и конкретно."""

# Gemini умеет анализировать изображения напрямую это плюс для нас
visual_response = gemini_model.generate_content([
    visual_prompt,
    {
        'mime_type': 'image/jpeg',
        'data': frame_to_base64(before_click['frame'])
    },
    "Кадр после клика:",
    {
        'mime_type': 'image/jpeg',
        'data': frame_to_base64(after_click['frame'])
    }
])

print("🖼️ ВИЗУАЛЬНОЕ СРАВНЕНИЕ:")
print(visual_response.text)

🔍 STAGE 1: Florence-2 перцепция...

[0.0s] Change: 0.0 | The image is a screenshot of a pop-up window with a message in Russian that reads "Send Registration...
[1.0s] Change: 0.3 | The image is a screenshot of a pop-up window titled "Send Registration Form". The window has a white...
[2.0s] Change: 0.6 | The image is a screenshot of a pop-up window with the text "Send Registration Form" in Russian. The ...
[3.0s] Change: 0.3 | The image is a screenshot of a pop-up window with a message that reads "Send Registration Form". The...
[4.0s] Change: 0.0 | The image is a screenshot of a pop-up window with a message that reads "Send Registration Form". The...
[5.0s] Change: 17.9 | The image is a screenshot of a pop-up window with a message in Russian that reads "Send Registration...
[6.0s] Change: 0.1 | The image is a screenshot of a pop-up window titled "Send Registration Form". The window has a white...
[7.0s] Change: 0.1 | The image is a screenshot of a pop-up window with a message in Russ